In [25]:
import pandas as pd
import numpy as np
import joblib

#Load data
df = pd.read_csv('../data/raw/Telco_customer_churn.csv')

print('---- Data Preprocessing ----')

#1 Handle target variable (Convert to binary)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print(f"Target variable: 0=No Churn, 1=Churn")

#2 Drop unnecessary columns
df = df.drop(['customerID'], axis=1)
print(f"Dropped costumerID (not predictive)")

#3 Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

#Remove target from numerics cols
numeric_cols = [col for col in numeric_cols if col != 'Churn']

print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

#4 Handle missing values (if any)
print("\n---- Missing Values ----")
print(df.isnull().sum())
#If missing: df.fillna(df.mean()) for numeric, df.fillna('Unknown') for categorical

#5 Encode categorical variables
print("\n---- Encoding Categorical Variables ----")

#Binary categorical (2 unique values)
binary_cols = {
    'gender': {'Male': 1, 'Female': 0},
    'Partner': {'Yes': 1, 'No': 0},
    'Dependents': {'Yes': 1, 'No': 0},
    'PhoneService': {'Yes': 1, 'No': 0},
    'PaperlessBilling': {'Yes': 1, 'No': 0},

    'OnlineSecurity': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'OnlineBackup': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'DeviceProtection': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'TechSupport': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'StreamingTV': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'StreamingMovies': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'MultipleLines': {'Yes': 1, 'No': 0, 'No phone service': 0}
}

for col, mapping in binary_cols.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

#Multi-class categorical (3 unique values) - one hot encoding
multi_cols = ['InternetService', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, prefix=multi_cols)

print("Encoded all categorical variables")

#6 Create interaction features
print("\n---- Creating interaction features ----")

#Tenure categories
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 36, 72],
                            labels=['0-1yr', '1-2yr', '2-3yr', '3+yr'])
df = pd.get_dummies(df, columns=['tenure_group'], drop_first=True)

#Charges per month ratio (high charges relative to tenure)
df['high_charges_ratio'] = df['MonthlyCharges'] / (df['tenure'] + 1)

#Convert column to a numeric value
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

#Total charges indicator
df['high_total_charges'] = (df['TotalCharges'] > df['TotalCharges'].median()).astype(int)

print("Created interaction features")

#Update feature lists
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'Churn']

print(f"\n Final feature count: {len(numeric_cols)} numeric + {len(df.columns) - len(numeric_cols) - 1} categorical")
print(f"Total features: {len(df.columns) - 1}")

print("\nRemaining object columns:")
print(df.select_dtypes(include=['object']).columns.tolist())

print("\nPrueba")
print(df.isnull().sum()[df.isnull().sum() > 0])

#Save processed data
df.to_csv('../data/processed/featured_churn_data.csv', index=False)
print(f"\n Data saves to featured_churn_data.csv")
print(f"Dataset shape: {df.shape}")




---- Data Preprocessing ----
Target variable: 0=No Churn, 1=Churn
Dropped costumerID (not predictive)

Numeric columns (3): ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Categorical columns (16): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']

---- Missing Values ----
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

---- Encoding Categorical Variables ----
Encoded all categorical var